# Практика 06. Трансформеры.

## «Улица Сезам и первые модели»

Когда говорят про первые **популярные** трансформер‑модели для NLP, часто вспоминают «улицу Сезам» — из‑за названий вроде ELMo, BERT, ERNIE, Grover, BigBird и т.п. Это реальные персонажи из детского шоу *Sesame Street*, а не просто случайные буквы.

Исторически цепочка в общих четрах выглядела примерно так:

- **ELMo (2018)** — Embeddings from Language Models.  
  Это ещё не трансформер, а двухслойный BiLSTM, но важная идея: получать **контекстные** эмбеддинги слов, которые зависят от предложения целиком, а не один фиксированный вектор на слово как в word2vec. ELMo показал, что предварительно обученный языковой моделью «большой кусок сети» можно потом дообучать под конкретные задачи и получать большой прирост качества.

- **ULMFiT / GPT (2018)** — первые массовые истории про «предобучили языковую модель → дообучили под задачу».  
  OpenAI GPT уже использует **только декодер‑часть трансформера** (causal LM), то есть по сути тот же механизм внимания, который мы разбираем, но в авторегрессионном режиме «предсказывай следующее слово».

- **BERT (2018)** — Bidirectional Encoder Representations from Transformers.  
  Это «чистый» **энкодер‑трансформер**, обученный на задаче masked language modeling: часть токенов в предложении маскируется, модель должна восстановить их по контексту слева и справа. BERT стал базовой рабочей лошадкой для кучи классических NLP‑задач: классификация текста, NER, QA, парсинг и т.д., потому что достаточно добавить сверху один простой слой и дообучить модель на своём датасете.

- Дальше пошёл зоопарк вариаций: RoBERTa, XLNet, ERNIE, BigBird и т.п.  
  Архитектурно это всё те же блоки encoder/decoder, self‑attention, multi‑head и FFN, но меняются **режимы предобучения, данные и трюки обучения**.

- Когда вы берёте `bert-base-uncased` из HuggingFace, под капотом там именно **энкодер‑часть трансформера**, почти один в один как в схемах выше.
- Когда вы берёте `gpt2`, это уже **декодер‑трансформер** с masked self‑attention (causal), который генерирует текст по одному токену.
- Все эти «модели с улицы Сезам» отличаются не магическими новыми слоями, а в основном:
  - тем, **какую часть трансформера** они используют (encoder / decoder / encoder+decoder),
  - **какую предобучающую задачу** решают (masked LM, causal LM, next sentence prediction и т.п.),
  - **как потом дообучаются** на конкретную задачу.

На практике это значит: разобравшись с устройством базового трансформера, вы автоматически понимаете, как устроены BERT, GPT и их многочисленные родственники.

## Какой трансформер под какую задачу?

Чтобы не теряться в зоопарке моделей, удобно мыслить не названиями, а **архитектурой и типом задачи**.

### Классические рабочие лошадки

| Модель / класс | Архитектура | Что умеет лучше всего | Типичные задачи |
|----------------|------------|------------------------|-----------------|
| **BERT и семья (RoBERTa, DistilBERT, mBERT, RuBERT, DeBERTa и т.п.)** | Только **энкодер** трансформера, двунаправленный self-attention (видит контекст слева и справа). | Понимать текст: строить богатые эмбеддинги предложений / токенов. Отлично подходит для задач «прочитай и пойми». | Классификация текста (токсичность, тональность, темы), NER, извлечение фактов, вопрос–ответ по документу, семантический поиск / эмбеддинги для поиска. |
| **GPT‑подобные (GPT‑2, GPT‑3, современные LLM)** | Только **декодер** трансформера, masked (causal) self-attention. | Генерировать текст по префиксу: продолжения, диалоги, код, стилизация. | Чат‑боты, генерация ответов, автодополнение текста и кода, свободная генерация, суммаризация, перевод, «ассистент‑универсал».|
| **Seq2Seq‑модели на трансформерах (Transformer, BART, T5 и т.п.)** | Полный **encoder–decoder**: энкодер читает вход, декодер генерирует выход. | Преобразовывать одну последовательность в другую. | Машинный перевод, суммаризация, перефразирование, генерация заголовков, style transfer и любые «input → output» задачи.

### Практический ориентиp для проектов

- Нужно **понять** текст и выдать один или несколько лейблов?  
  Берём BERT‑подобную модель и добавляем сверху простой классификатор.

- Нужно **извлечь** сущности/ответы из текста (NER, QA)?  
  Тоже хорошо подходят BERT‑семейство и его аналоги: архитектура «энкодер + маленькая голова» даёт сильную базу на понимание.

- Нужно **написать/продолжить** текст, вести диалог, сгенерировать код?  
  Здесь лучше чувствуют себя GPT‑подобные декодеры: они обучены именно на задаче «предсказывать следующее слово» и умеют поддерживать связный текст.

- Нужно **преобразовать** один текст в другой (перевод, суммаризация)?  
  Удобны Seq2Seq‑модели (BART, T5 и их аналоги), где энкодер читает вход, а декодер пошагово генерирует выход.

В практической части мы будем работать именно с такими готовыми предобученными моделями из `transformers`: по сути вы будете брать один из этих «шаблонов» (энкодер, декодер или encoder–decoder) и слегка дообучать/доиспользовать его под свою задачу.

## Решаем задачу про `токсичность`

В этой практике мы работаем с моделью `cointegrated/rubert-tiny2` — это **русскоязычный BERT‑подобный энкодер**.  
Мы берём его «как есть» и добавляем сверху простой классификатор, чтобы решать задачу мультилейбл‑классификации токсичных комментариев.

Связь с теорией:

- Внутри `rubert-tiny2` — те самые блоки **self‑attention + FFN + LayerNorm + skip‑connections**, которые мы рассмотрели на лекции.
- Self‑attention позволяет каждому токену в комментарии «посмотреть» на остальные слова и понять контекст: сарказм, оскорбление, угрозу и т.п.
- Энкодер собирает информацию по всей последовательности в скрытые представления; затем мы берём [CLS]‑вектор (или пуллинг по токенам) и подаём его в небольшой линейный слой, который предсказывает 4 метки: `INSULT`, `OBSCENITY`, `THREAT`, `NORMAL`.

То есть на практике мы и будем делать ровно то, для чего BERT‑семейство и создавалось:  
**предобученный трансформер как универсальный «пониматель текста» + тонкий слой под конкретную задачу.**

### Какие метки мы предсказываем и что в них сложного

| Метка          | Что примерно означает                                 | Что делает задачу нетривиальной |
|----------------|--------------------------------------------------------|---------------------------------|
| `__label__NORMAL`    | Нейтральный или обычный комментарий                    | Нужна **устойчивость к шуму**: текст может содержать грубые слова в цитатах или нейтральном контексте. |
| `__label__INSULT`    | Оскорбление, унижение человека / группы                | Часто выражается эвфемизмами, сарказмом, намёками; важно **понимать контекст**, а не только отдельные слова. |
| `__label__OBSCENITY` | Ненормативная лексика, мат                            | Формально легко по словарям, но модель должна отличать **буквальное употребление** от цитат, шуток и т.п. |
| `__label__THREAT`    | Прямая или завуалированная угроза                      | Часто завуалирована («поговорим лично», «я тебя найду»), требуется **учёт смысла фразы целиком**. |

Почему здесь особенно полезен трансформер‑энкодер:

- Модель видит комментарий **целиком** через двунаправленный self‑attention и может учитывать, к кому обращено сообщение и в каком тоне оно сказано.
- Мультилейбл‑формат (комбинации INSULT+THREAT, INSULT+OBSCENITY и т.п.) показывает, что реальные комментарии часто содержат **смешанные типы токсичности**, а не один класс — это лучше моделируется многомерным выходом поверх общих текстовых признаков.

In [11]:
import kagglehub

path = kagglehub.dataset_download("alexandersemiletov/toxic-russian-comments")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\mapks\.cache\kagglehub\datasets\alexandersemiletov\toxic-russian-comments\versions\1


Про датасет можно почитать по ссылке: https://www.kaggle.com/code/alexandersemiletov/starter-read-toxic-russian-comments-dataset

In [13]:
import re
import pandas as pd
from pathlib import Path
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer

txt_file = next(Path(path).glob("*.txt"))

data = []
label_counter = Counter()

with open(txt_file, encoding="utf-8") as f:
    for line in f:
        raw_labels = re.findall(r'__label__\S+', line)
        labels = []
        for raw_label in raw_labels:
            labels.extend(raw_label.split(','))

        text = re.sub(r'__label__\S+\s*', '', line).strip()
        data.append((text, labels))
        label_counter.update(labels)

df = pd.DataFrame(data, columns=["text", "labels"])

# Преобразование меток в мультилейбл формат
mlb = MultiLabelBinarizer()
binary_labels = mlb.fit_transform(df['labels'])
label_names = mlb.classes_

print("Уникальные метки:", label_names)
print("\nПример преобразования:")
for i in range(min(3, len(df))):
    print(f"Текст: {df['text'].iloc[i]}")
    print(f"Метки: {df['labels'].iloc[i]}")
    print(f"Бинарный вектор: {binary_labels[i]}")
    print("---")

print("\nСтатистика по меткам:")
for label, count in label_counter.most_common():
    print(f"{label}: {count} примеров")

print("\nСтатистика по комбинациям меток:")
combinations_counter = Counter([tuple(sorted(labels)) for labels in df['labels']])
for combo, count in combinations_counter.most_common():
    print(f"{combo}: {count} примеров")

print("\nВсе уникальные метки:")
for i, label in enumerate(label_names, 1):
    print(f"{i}. {label}")


Уникальные метки: ['__label__INSULT' '__label__NORMAL' '__label__OBSCENITY'
 '__label__THREAT']

Пример преобразования:
Текст: скотина! что сказать
Метки: ['__label__INSULT']
Бинарный вектор: [1 0 0 0]
---
Текст: я сегодня проезжала по рабочей и между домами снитенко и гомолысовой магазином ( на пустыре) бежала кошка похожего окраса. может, я и ошиблась, но необычный окрас бросился в глаза.
Метки: ['__label__NORMAL']
Бинарный вектор: [0 1 0 0]
---
Текст: очередной лохотрон. зачем придумывать очередной налог на воздух, если можно обьявить инсульт и грипп- пандемией! и лихо на придурках зарабатывать годами на штрафах, фейковых вакцинах, всевозможных платных тестах, продажей масок и перчаток по баснословным ценам.. самое смешное, что бараны блеют и верят пастуху, телевизору. живут как под гипнозом. не думая, не глядя по сторонам.
Метки: ['__label__NORMAL']
Бинарный вектор: [0 1 0 0]
---

Статистика по меткам:
__label__NORMAL: 203685 примеров
__label__INSULT: 36826 примеров
__label__THREAT

In [14]:
import re
import pandas as pd
from pathlib import Path
from collections import Counter
import kagglehub
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from torch.utils.data import Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.preprocessing import MultiLabelBinarizer
from torch.nn import BCEWithLogitsLoss

train_val_df, test_df = train_test_split(df, test_size=0.2, stratify=df['labels'], random_state=555)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, stratify=train_val_df['labels'], random_state=555)
# (0.25 от 0.8 = 0.2 -> итог: 60% train, 20% val, 20% test)

print("\n=== Проверка структуры данных ===")
print(f"Всего меток: {len(label_names)}")
print("Примеры меток и их распределение:")
print(df[label_names].sum())

multilabel_counts = (df[label_names].sum(axis=1) > 1).sum()
print(f"\nКоличество примеров с несколькими метками: {multilabel_counts} ({multilabel_counts/len(df):.1%})")

class ToxicCommentsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

tokenizer = BertTokenizer.from_pretrained('cointegrated/rubert-tiny2')
model = BertForSequenceClassification.from_pretrained(
    'cointegrated/rubert-tiny2',
    num_labels=len(label_names),
    problem_type="multi_label_classification"
)

# Подсчёт весов для дисбаланса и увеличения веса для редких классов
# Вычисляем веса для каждого класса
class_counts = df[label_names].sum().values
total_samples = len(df)
class_weights = total_samples / (len(label_names) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32)
print("Class weights:", class_weights)

train_dataset = ToxicCommentsDataset(
    train_df['text'].tolist(),
    train_df[label_names].values,
    tokenizer
)
val_dataset = ToxicCommentsDataset(
    val_df['text'].tolist(),
    val_df[label_names].values,
    tokenizer
)

def find_optimal_threshold(labels, logits):
    """Находит оптимальный порог для каждого класса по F1-score"""
    thresholds = []
    for i in range(logits.shape[1]):
        preds = torch.sigmoid(torch.tensor(logits[:, i]))
        best_th = 0.5
        best_f1 = 0
        for th in np.arange(0.3, 0.7, 0.05):
            f1 = f1_score(labels[:, i], (preds > th).int())
            if f1 > best_f1:
                best_f1 = f1
                best_th = th
        thresholds.append(best_th)
    return np.array(thresholds)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    thresholds = find_optimal_threshold(labels, logits)
    preds = torch.sigmoid(torch.tensor(logits)) > torch.tensor(thresholds)

    print("\nПримеры предсказаний:")
    for i in range(min(3, len(labels))):
        true_labels = [label_names[idx] for idx, val in enumerate(labels[i]) if val == 1]
        pred_labels = [label_names[idx] for idx, val in enumerate(preds[i]) if val]
        print(f"\nПример {i+1}:")
        print(f"Истинные метки: {true_labels}")
        print(f"Предсказанные метки: {pred_labels}")
        print(f"Пороги: {thresholds}")

    return {
        'f1_micro': f1_score(labels, preds, average='micro'),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'precision_macro': precision_score(labels, preds, average='macro'),
        'recall_macro': recall_score(labels, preds, average='macro'),
        'accuracy': accuracy_score(labels, preds),
        'thresholds': thresholds.tolist()
    }

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch: int = 16):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = BCEWithLogitsLoss(pos_weight=self.class_weights.to(model.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir='./results',
    run_name='toxic-comment-sercher',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

trainer.train()

model.save_pretrained('./toxic_comment_model')
tokenizer.save_pretrained('./toxic_comment_model')


=== Проверка структуры данных ===
Всего меток: 4
Примеры меток и их распределение:


KeyError: "None of [Index(['__label__INSULT', '__label__NORMAL', '__label__OBSCENITY',\n       '__label__THREAT'],\n      dtype='object')] are in the [columns]"

### Зачем настраивать пороги в мультилейбл‑классификации

В мультилейбл‑задачах модель на выходе даёт **не метки**, а вероятности по каждой метке:  
для нашего случая — 4 числа от 0 до 1: `p_INSULT`, `p_NORMAL`, `p_OBSCENITY`, `p_THREAT`.

Чтобы превратить их в 0/1, нам нужен порог по каждой метке:

- если `p_label ≥ threshold_label` → считаем, что метка присутствует;
- иначе → считаем, что её нет.

Почему нельзя просто взять «по дефолту 0.5»:

- Классы сильно **несбалансированы**: `NORMAL` встречается гораздо чаще, чем `OBSCENITY` или комбинированные метки.
- Ошибка пропуска редкой токсичной метки (например, угрозы) может быть важнее, чем ложное срабатывание.
- Разные метки имеют **разные распределения вероятностей**: для каких‑то модель уверена даже при 0.8, а для других обычно даёт более «приглушённые» значения.

Поэтому разумно:

- Подобрать отдельный порог для каждой метки, который даёт лучший баланс между **precision** и **recall** (например, по F1‑макро или по F1 для конкретной метки).
- Делать это на валидационной выборке, перебирая пороги в диапазоне вроде `[0.1, 0.9]` с шагом.

Интуитивно:

- Чем **ниже** порог → тем больше мы «подозреваем токсичность»: растёт recall, но падает precision.
- Чем **выше** порог → тем более «осторожно» ставим метку: растёт precision, но можем пропускать реальные случаи (падает recall).

В эксперименте мы как раз подбираем пороги по валидации и видим, как это влияет на F1‑микро, F1‑макро и метрики по отдельным меткам — это стандартный практический шаг в мультилейбл‑классификации поверх трансформеров.

In [ ]:
import matplotlib.pyplot as plt

# Извлечение истории обучения
log_history = trainer.state.log_history

train_epochs = []
train_loss = []

eval_epochs = []
eval_loss = []
f1_macro = []
precision_macro = []
recall_macro = []

for entry in log_history:
    if 'loss' in entry and 'epoch' in entry:
        train_loss.append(entry['loss'])
        train_epochs.append(entry['epoch'])
    if 'eval_loss' in entry:
        eval_loss.append(entry['eval_loss'])
        eval_epochs.append(entry['epoch'])
        f1_macro.append(entry.get('eval_f1_macro'))
        precision_macro.append(entry.get('eval_precision_macro'))
        recall_macro.append(entry.get('eval_recall_macro'))

plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(train_epochs, train_loss, label='Train Loss', marker='o')
plt.plot(eval_epochs, eval_loss, label='Eval Loss', marker='x')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Train/Eval Loss Over Epochs')
plt.legend()
plt.grid(True)

plt.subplot(2, 2, 2)
plt.plot(eval_epochs, f1_macro, label='F1 Macro', marker='o')
plt.xlabel('Epoch')
plt.ylabel('F1 Macro')
plt.title('F1 Macro Over Epochs')
plt.grid(True)

plt.subplot(2, 2, 3)
plt.plot(eval_epochs, precision_macro, label='Precision Macro', color='green', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Precision Macro')
plt.title('Precision Macro Over Epochs')
plt.grid(True)

plt.subplot(2, 2, 4)
plt.plot(eval_epochs, recall_macro, label='Recall Macro', color='red', marker='^')
plt.xlabel('Epoch')
plt.ylabel('Recall Macro')
plt.title('Recall Macro Over Epochs')
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import torch
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from itertools import combinations
from collections import defaultdict

def detailed_label_analysis(true_labels, pred_labels, label_names):
    results = {}

    print("\n" + "="*50)
    print("Метрики по каждому лейблу:")
    print("="*50)
    report = classification_report(true_labels, pred_labels, target_names=label_names, output_dict=True, zero_division=0)
    for label in label_names:
        results[f"label_{label}"] = report[label]
        print(f"\nЛейбл: {label}")
        print(f"Precision: {report[label]['precision']:.3f}")
        print(f"Recall: {report[label]['recall']:.3f}")
        print(f"F1-score: {report[label]['f1-score']:.3f}")
        print(f"Поддержка: {report[label]['support']}")

    print("\n" + "="*50)
    print("Метрики по комбинациям лейблов:")
    print("="*50)

    # Собираем все встречающиеся комбинации в тестовых данных
    unique_combinations = set(tuple(sorted(np.where(row)[0])) for row in true_labels if sum(row) > 1)

    for combo in unique_combinations:
        if len(combo) < 2:
            continue

        combo_name = "+".join(label_names[i] for i in combo)

        # True Positive: случаи, где и предсказано, и истинно
        tp_mask = np.all(true_labels[:, combo] == 1, axis=1) & np.all(pred_labels[:, combo] == 1, axis=1)
        tp = sum(tp_mask)

        # False Positive: предсказано, но не истинно
        fp_mask = np.all(pred_labels[:, combo] == 1, axis=1) & ~np.all(true_labels[:, combo] == 1, axis=1)
        fp = sum(fp_mask)

        # False Negative: истинно, но не предсказано
        fn_mask = np.all(true_labels[:, combo] == 1, axis=1) & ~np.all(pred_labels[:, combo] == 1, axis=1)
        fn = sum(fn_mask)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall + 1e-9)

        results[f"combo_{combo_name}"] = {
            'precision': precision,
            'recall': recall,
            'f1-score': f1,
        }

        print(f"\nКомбинация: {combo_name}")
        print(f"Precision: {precision:.3f}")
        print(f"Recall: {recall:.3f}")
        print(f"F1-score: {f1:.3f}")

    return results

def compute_metrics(eval_pred):
    logits, true_labels = eval_pred
    thresholds = find_optimal_threshold(true_labels, logits)
    pred_probs = torch.sigmoid(torch.tensor(logits)).numpy()
    pred_labels = (pred_probs > thresholds).astype(int)

    detailed_metrics = detailed_label_analysis(true_labels, pred_labels, label_names)

    metrics = {
        'accuracy': accuracy_score(true_labels, pred_labels),
        'f1_micro': f1_score(true_labels, pred_labels, average='micro'),
        'f1_macro': f1_score(true_labels, pred_labels, average='macro'),
        'precision_macro': precision_score(true_labels, pred_labels, average='macro'),
        'recall_macro': recall_score(true_labels, pred_labels, average='macro'),
        'thresholds': thresholds.tolist(),
    }

    metrics.update(detailed_metrics)

    return metrics

test_dataset = ToxicCommentsDataset(
    test_df['text'].tolist(),
    test_df[label_names].values,
    tokenizer
)

# После обучения получаем предсказания на валидационном наборе
val_predictions = trainer.predict(test_dataset)
val_logits = val_predictions.predictions
val_true_labels = val_predictions.label_ids

thresholds = find_optimal_threshold(val_true_labels, val_logits)
val_pred_probs = torch.sigmoid(torch.tensor(val_logits)).numpy()
val_pred_labels = (val_pred_probs > thresholds).astype(int)

detailed_label_analysis(val_true_labels, val_pred_labels, label_names)


In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.utils import resample
from torch import nn

print("Уникальные метки:", label_names)
print("\nПример преобразования:")
for i in range(min(3, len(df))):
    print(f"Текст: {df['text'].iloc[i]}")
    print(f"Метки: {df['labels'].iloc[i]}")
    print(f"Бинарный вектор: {binary_labels[i]}")
    print("---")

print("\nСтатистика по меткам:")
for label, count in label_counter.most_common():
    print(f"{label}: {count} примеров")

print("\nСтатистика по комбинациям меток:")
combinations_counter = Counter([tuple(sorted(labels)) for labels in df['labels']])
for combo, count in combinations_counter.most_common():
    print(f"{combo}: {count} примеров")

print("\nВсе уникальные метки:")
for i, label in enumerate(label_names, 1):
    print(f"{i}. {label}")


# Подготовка данных с увеличением редких комбинаций
def augment_rare_combinations(df, label_names, min_samples):
    """Увеличиваем количество примеров для редких комбинаций лейблов с логированием"""
    from itertools import combinations

    print("Начало аугментации редких комбинаций лейблов")
    print(f"Исходное количество примеров: {len(df)}")

    # Сначала найдем все реально существующие комбинации
    existing_combinations = set()
    for labels in df['labels']:
        if len(labels) >= 2:  # Нас интересуют только комбинации из 2+ лейблов
            existing_combinations.add(tuple(sorted(labels)))

    combo_stats = {}
    for combo in existing_combinations:
      if len(combo) in [2, 3]:  # Только пары и тройки
          mask = df['labels'].apply(lambda x: set(x) == set(combo))
          count = sum(mask)
          if count < min_samples:
              combo_stats[combo] = count
              print(f"Редкая комбинация: {'+'.join(combo)} - {count} примеров (требуется минимум {min_samples})")

    augmented_dfs = []
    augmentation_report = []

    for combo, count in combo_stats.items():
        if count == 0:
            continue

        mask = df['labels'].apply(lambda x: set(combo).issubset(x))
        samples = df[mask]

        n_samples = min(min_samples - count, count*4)
        if n_samples <= 0:
            continue

        augmented = resample(samples,
                          replace=True,
                          n_samples=n_samples,
                          random_state=42)
        augmented_dfs.append(augmented)

        augmentation_report.append({
            'combination': '+'.join(combo),
            'original_count': count,
            'new_samples': n_samples,
            'total_after_aug': count + n_samples
        })

    if augmented_dfs:
        result_df = pd.concat([df] + augmented_dfs)

        print("\nОтчет по аугментации:")
        print("{:<60} {:<15} {:<15} {:<15}".format(
            "Комбинация", "Было", "Добавлено", "Стало"))
        print("-"*105)

        for item in augmentation_report:
            print("{:<60} {:<15} {:<15} {:<15}".format(
                item['combination'],
                item['original_count'],
                item['new_samples'],
                item['total_after_aug']))

        print(f"\nОбщее количество примеров после аугментации: {len(result_df)}")
        print(f"Добавлено примеров: {len(result_df) - len(df)}")

        return result_df

    print("Не было найдено редких комбинаций для аугментации")
    return df

df_augmented = augment_rare_combinations(df, label_names, min_samples=500)

In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.utils import resample
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from torch import nn
from torch.nn import BCEWithLogitsLoss
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import EarlyStoppingCallback

if isinstance(label_names, np.ndarray):
    label_names = label_names.tolist()

train_val_df_augmented, test_df_augmented = train_test_split(df_augmented, test_size=0.2, stratify=df_augmented['labels'], random_state=555)
train_df_augmented, val_df_augmented = train_test_split(train_val_df_augmented, test_size=0.25, stratify=train_val_df_augmented['labels'], random_state=555)
# (0.25 от 0.8 = 0.2 -> итог: 60% train, 20% val, 20% test)

train_dataset_augmented = ToxicCommentsDataset(
    train_df_augmented['text'].tolist(),
    train_df_augmented[label_names].values,
    tokenizer
)
val_dataset_augmented = ToxicCommentsDataset(
    val_df_augmented['text'].tolist(),
    val_df_augmented[label_names].values,
    tokenizer
)

# Модифицируем архитектуру для учета взаимодействий лейблов
class FineTunedBertForMultiLabel(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        # Добавляем слои для взаимодействия лейблов
        self.interaction = nn.Sequential(
            nn.Linear(config.num_labels, 64),
            nn.ReLU(),
            nn.Linear(64, config.num_labels)
        )

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
      outputs = super().forward(
          input_ids=input_ids,
          attention_mask=attention_mask,
          labels=labels,
          **kwargs
      )

      if hasattr(outputs, 'logits'):
          logits = outputs.logits
          interaction = torch.sigmoid(self.interaction(torch.sigmoid(logits)))
          outputs.logits = logits + 0.3 * interaction

      return outputs

important_combinations = [
    [label_names.index('__label__OBSCENITY'), label_names.index('__label__THREAT')],
    [label_names.index('__label__INSULT'), label_names.index('__label__OBSCENITY')],
    [label_names.index('__label__INSULT'), label_names.index('__label__THREAT')],
    [label_names.index('__label__INSULT'), label_names.index('__label__OBSCENITY'),
     label_names.index('__label__THREAT')]
]

# Функция для вычисления метрик с акцентом на комбинации
def compute_metrics_with_combos(eval_pred):
    logits, labels = eval_pred
    preds = (torch.sigmoid(torch.tensor(logits)) > thresholds).int().numpy()

    # Базовые метрики
    metrics = {
        'f1_micro': f1_score(labels, preds, average='micro'),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'precision_macro': precision_score(labels, preds, average='macro'),
        'recall_macro': recall_score(labels, preds, average='macro'),
    }

    # Метрики для комбинаций
    for i, combo in enumerate(important_combinations):
        combo_name = "+".join(label_names[idx] for idx in combo)
        mask = np.all(labels[:, combo] == 1, axis=1)

        if sum(mask) > 0:
            y_true = labels[mask]
            y_pred = preds[mask]

            # Для комбинации считаем правильным предсказание всех лейблов
            correct = np.all(y_pred[:, combo] == 1, axis=1)
            precision = np.mean(correct)

            metrics[f'combo_{combo_name}_precision'] = precision
            metrics[f'combo_{combo_name}_recall'] = precision  # Для комбинаций precision=recall
            metrics[f'combo_{combo_name}_support'] = sum(mask)
    return metrics

class CombinationLoss(BCEWithLogitsLoss):
    def __init__(self, important_combos=None, combo_weight=3.0, **kwargs):
        super().__init__(**kwargs)
        self.important_combos = important_combos or []
        self.combo_weight = combo_weight

    def forward(self, input, target):
        # Базовая loss
        loss = super().forward(input, target)

        # Дополнительный штраф за ошибки в важных комбинациях
        if self.important_combos:
            combo_loss = 0
            for combo in self.important_combos:
                combo_mask = torch.all(target[:, combo] == 1, dim=1)
                if combo_mask.any():
                    # Берем только нужные лейблы из комбинации
                    combo_input = input[combo_mask][:, combo]
                    combo_target = target[combo_mask][:, combo]

                    # Вычисляем loss только для лейблов в текущей комбинации
                    combo_loss += BCEWithLogitsLoss()(combo_input, combo_target)

            loss += self.combo_weight * (combo_loss / len(self.important_combos))

        return loss

# Сделал кастомный Trainer для того чтобы сделать акцент на важных комбинациях
class CustomTrainer(Trainer):
    def __init__(self, *args, important_combos=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.important_combos = important_combos or []

    def compute_loss(self, model, inputs, return_outputs=False,num_items_in_batch=16):
      labels = inputs.pop("labels")
      outputs = model(**inputs)
      logits = outputs.logits

      weights = torch.ones_like(labels)
      for combo in self.important_combos:
          combo_mask = torch.all(labels[:, combo] == 1, dim=1)
          weights[combo_mask] = 2.0

      loss_fct = CombinationLoss(
          important_combos=self.important_combos,
          pos_weight=weights.mean(dim=0)
      )
      loss = loss_fct(logits, labels)

      return (loss, outputs) if return_outputs else loss

model = FineTunedBertForMultiLabel.from_pretrained(
    "./toxic_comment_model",
    num_labels=len(label_names),
    problem_type="multi_label_classification"
)

training_args = TrainingArguments(
    output_dir='./fine_tuned_model',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy='epoch',
    eval_steps=200,
    save_strategy='epoch',
    logging_dir='./logs',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='loss',
    learning_rate=3e-6,
    weight_decay=0.05,
    gradient_accumulation_steps=4
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_augmented,
    eval_dataset=val_dataset_augmented,
    compute_metrics=compute_metrics_with_combos,
    important_combos=important_combinations,
)

trainer.add_callback(EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.01
))


trainer.train()

model.save_pretrained('./fine_tuned_model/combo_focused')
tokenizer.save_pretrained('./fine_tuned_model/combo_focused')

In [ ]:
import matplotlib.pyplot as plt

# Извлечение истории обучения
log_history = trainer.state.log_history

train_epochs = []
train_loss = []

eval_epochs = []
eval_loss = []
f1_macro = []
precision_macro = []
recall_macro = []

for entry in log_history:
    if 'loss' in entry and 'epoch' in entry:
        train_loss.append(entry['loss'])
        train_epochs.append(entry['epoch'])
    if 'eval_loss' in entry:
        eval_loss.append(entry['eval_loss'])
        eval_epochs.append(entry['epoch'])
        f1_macro.append(entry.get('eval_f1_macro'))
        precision_macro.append(entry.get('eval_precision_macro'))
        recall_macro.append(entry.get('eval_recall_macro'))

plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(train_epochs, train_loss, label='Train Loss', marker='o')
plt.plot(eval_epochs, eval_loss, label='Eval Loss', marker='x')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Train/Eval Loss Over Epochs')
plt.legend()
plt.grid(True)

plt.subplot(2, 2, 2)
plt.plot(eval_epochs, f1_macro, label='F1 Macro', marker='o')
plt.xlabel('Epoch')
plt.ylabel('F1 Macro')
plt.title('F1 Macro Over Epochs')
plt.grid(True)

plt.subplot(2, 2, 3)
plt.plot(eval_epochs, precision_macro, label='Precision Macro', color='green', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Precision Macro')
plt.title('Precision Macro Over Epochs')
plt.grid(True)

plt.subplot(2, 2, 4)
plt.plot(eval_epochs, recall_macro, label='Recall Macro', color='red', marker='^')
plt.xlabel('Epoch')
plt.ylabel('Recall Macro')
plt.title('Recall Macro Over Epochs')
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import torch
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
from itertools import combinations
from collections import defaultdict

def detailed_label_analysis(true_labels, pred_labels, label_names):
    results = {}

    print("\n" + "="*50)
    print("Метрики по каждому лейблу:")
    print("="*50)
    report = classification_report(true_labels, pred_labels, target_names=label_names, output_dict=True, zero_division=0)
    for i, label in enumerate(label_names):
        precision = report[label]['precision']
        recall = report[label]['recall']
        f1 = report[label]['f1-score']

        correct = np.sum(true_labels[:, i] == pred_labels[:, i])
        total = len(true_labels)
        accuracy = correct / total

        results[f"label_{label}"] = {
            'precision': precision,
            'recall': recall,
            'f1-score': f1,
            'accuracy': accuracy
        }

        print(f"\nЛейбл: {label}")
        print(f"Precision: {precision:.3f}")
        print(f"Recall: {recall:.3f}")
        print(f"F1-score: {f1:.3f}")
        print(f"Accuracy: {accuracy:.3f}")
    print("\n" + "="*50)
    print("Метрики по комбинациям лейблов:")
    print("="*50)

    # Собираем все встречающиеся комбинации в тестовых данных
    unique_combinations = set(tuple(sorted(np.where(row)[0])) for row in true_labels if sum(row) > 1)

    for combo in unique_combinations:
        if len(combo) < 2:
            continue

        combo_name = "+".join(label_names[i] for i in combo)

        # True Positive: случаи, где и предсказано, и истинно
        tp_mask = np.all(true_labels[:, combo] == 1, axis=1) & np.all(pred_labels[:, combo] == 1, axis=1)
        tp = sum(tp_mask)

        # False Positive: предсказано, но не истинно
        fp_mask = np.all(pred_labels[:, combo] == 1, axis=1) & ~np.all(true_labels[:, combo] == 1, axis=1)
        fp = sum(fp_mask)

        # False Negative: истинно, но не предсказано
        fn_mask = np.all(true_labels[:, combo] == 1, axis=1) & ~np.all(pred_labels[:, combo] == 1, axis=1)
        fn = sum(fn_mask)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall + 1e-9)

        combo_true_mask = np.all(true_labels[:, combo] == 1, axis=1)
        combo_pred_mask = np.all(pred_labels[:, combo] == 1, axis=1)
        correct_mask = combo_true_mask & combo_pred_mask
        accuracy = sum(correct_mask) / sum(combo_true_mask) if sum(combo_true_mask) > 0 else 0.0

        results[f"combo_{combo_name}"] = {
            'precision': precision,
            'recall': recall,
            'f1-score': f1,
            'accuracy': accuracy
        }

        print(f"\nКомбинация: {combo_name}")
        print(f"Precision: {precision:.3f}")
        print(f"Recall: {recall:.3f}")
        print(f"F1-score: {f1:.3f}")
        print(f"Accuracy: {accuracy:.3f}")

    return results

def compute_metrics(eval_pred):
    logits, true_labels = eval_pred
    thresholds = find_optimal_threshold(true_labels, logits)
    pred_probs = torch.sigmoid(torch.tensor(logits)).numpy()
    pred_labels = (pred_probs > thresholds).astype(int)

    detailed_metrics = detailed_label_analysis(true_labels, pred_labels, label_names)

    metrics = {
        'accuracy': accuracy_score(true_labels, pred_labels),
        'f1_micro': f1_score(true_labels, pred_labels, average='micro'),
        'f1_macro': f1_score(true_labels, pred_labels, average='macro'),
        'precision_macro': precision_score(true_labels, pred_labels, average='macro'),
        'recall_macro': recall_score(true_labels, pred_labels, average='macro'),
        'thresholds': thresholds.tolist(),
    }

    metrics.update(detailed_metrics)

    return metrics

test_dataset_augmented = ToxicCommentsDataset(
    test_df_augmented['text'].tolist(),
    test_df_augmented[label_names].values,
    tokenizer
)

# После обучения получаем предсказания на валидационном наборе
val_predictions = trainer.predict(test_dataset_augmented)
val_logits = val_predictions.predictions
val_true_labels = val_predictions.label_ids

thresholds = find_optimal_threshold(val_true_labels, val_logits)
val_pred_probs = torch.sigmoid(torch.tensor(val_logits)).numpy()
val_pred_labels = (val_pred_probs > thresholds).astype(int)

detailed_label_analysis(val_true_labels, val_pred_labels, label_names)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Загрузка модели и токенизатора
model_path = "./fine_tuned_model/combo_focused"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

class_names = mlb.classes_

# Интерфейс ввода
text_input = widgets.Text(
    value='',
    placeholder='Введите текст для классификации',
    description='Текст:',
    layout=widgets.Layout(width='70%')
)
output = widgets.Output()

def classify_text(change):
    output.clear_output()
    text = change['new'].strip()
    if not text:
        return

    # Токенизация и предсказание
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        pred_class = torch.argmax(probs, dim=1).item()

    # Вывод результата
    with output:
        print(f" Класс: {class_names[pred_class]}")
        print(f" Вероятности: {probs.numpy().round(3)}")


text_input.observe(classify_text, names='value')

display(text_input, output)

# Практическое задание.
Попробуйте обучить модель (можно первую, можно вторую, можно обе.).

Для более полного понимания работы моделей следует написать вывод confusion matrix для каждой из меток.
в мультилейбл‑задаче удобно представлять, будто у нас есть **четыре отдельных бинарных классификатора** (по одной матрице ошибок на каждую метку), плюс дополнительные агрегированные метрики (F1‑микро, F1‑макро), которые суммируют картину по всем меткам сразу.

Также необходимо выполнить ручную проверку...